# 59 — General Region-Constrained Design Theory

**Core statement**

Optimization selects points. Physics specifies regions.

This notebook generalizes region-constrained design from a photonic case study into a reusable engineering design theory.

## Runtime requirements

This notebook uses `numpy`, `pandas`, `matplotlib`, `scipy`, and `IPython`.

```bash
pip install numpy pandas matplotlib scipy ipython
```

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, Ellipse, Polygon
from scipy import ndimage
from IPython.display import FileLink, display

OUT = Path("outputs/notebook_59_region_constrained_design_theory")
FIG = OUT / "figures"
DATA = OUT / "data"
FIG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 11,
    "axes.titlesize": 18,
    "axes.labelsize": 12,
})

def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, bbox_inches="tight")
    return path

def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def box(ax, x, y, w, h, title, subtitle="", lw=2.0, rounded=True, title_size=11, subtitle_size=8):
    if rounded:
        patch = FancyBboxPatch((x,y), w,h, boxstyle="round,pad=0.02,rounding_size=0.025",
                               fill=False, linewidth=lw, edgecolor="black")
    else:
        patch = Rectangle((x,y), w,h, fill=False, linewidth=lw, edgecolor="black")
    ax.add_patch(patch)
    ax.text(x+w/2, y+h*0.62, title, ha="center", va="center", fontsize=title_size, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.27, subtitle, ha="center", va="center", fontsize=subtitle_size)
    return patch

def arrow(ax, start, end, lw=2.0):
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", linewidth=lw, shrinkA=0, shrinkB=0))

def chain_horizontal(ax, items, y=0.50, x0=0.04, x1=0.96, w=None, h=0.17, title_size=10, subtitle_size=8):
    n = len(items)
    if w is None:
        w = min(0.12, (x1-x0)/(n*1.25))
    xs = np.linspace(x0, x1-w, n)
    for i,(t,s) in enumerate(items):
        box(ax, xs[i], y, w, h, t, s, title_size=title_size, subtitle_size=subtitle_size)
        if i < n-1:
            arrow(ax, (xs[i]+w+0.01, y+h/2), (xs[i+1]-0.01, y+h/2), lw=2)
    return xs

def chain_vertical(ax, items, x=0.5, y0=0.78, dy=0.12, w=0.34, h=0.075, title_size=11):
    for i,(t,s) in enumerate(items):
        y = y0 - i*dy
        box(ax, x-w/2, y, w, h, t, s, title_size=title_size)
        if i < len(items)-1:
            arrow(ax, (x, y), (x, y-dy+h), lw=1.8)

# Synthetic region model for universal engineering examples.
N = 280
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, y)
threshold = 0.38

def sigmoid(z, s=0.045):
    return 1 / (1 + np.exp(-z/s))

def region_field(center=0.60, width=0.24, yscale=0.23, amp=1.0, bump=0.25, hole=False, split=False):
    plateau = sigmoid(X-(center-width), 0.045) * sigmoid((center+width)-X, 0.045)
    low_y = np.exp(-(Y/yscale)**1.55)
    shoulder = bump*np.exp(-((X-(center+0.18))**2/0.018 + (Y-0.08)**2/0.025))
    Z = amp*np.clip(plateau*low_y + shoulder, 0, 1)
    if hole:
        Z -= 0.55*np.exp(-((X-center)**2/0.006 + (Y-0.12)**2/0.006))
    if split:
        Z -= 0.70*np.exp(-((X-center)**2/0.004 + (Y-0.08)**2/0.04))
    return np.clip(Z, 0, 1)

def metrics(Z, threshold=threshold):
    mask = Z >= threshold
    labels, ncomp = ndimage.label(mask)
    if ncomp == 0:
        return dict(mask=mask, largest=np.zeros_like(mask), dist=np.zeros_like(Z), ncomp=0,
                    area=0.0, largest_area=0.0, max_margin=0.0, avg_margin=0.0, xstar=np.nan, ystar=np.nan)
    sizes = ndimage.sum(mask, labels, index=np.arange(1, ncomp+1))
    lid = int(np.argmax(sizes)+1)
    largest = labels == lid
    dist_px = ndimage.distance_transform_edt(largest)
    dist = dist_px / dist_px.max() if dist_px.max() else dist_px
    idx = np.unravel_index(np.argmax(dist_px), dist_px.shape)
    return dict(mask=mask, largest=largest, dist=dist, ncomp=int(ncomp),
                area=float(mask.mean()), largest_area=float(largest.mean()),
                max_margin=float(dist_px.max()/N), avg_margin=float(dist[largest].mean()) if largest.any() else 0.0,
                xstar=float(x[idx[1]]), ystar=float(y[idx[0]]))

base_Z = region_field()
base_M = metrics(base_Z)

## 00 Context

What structure exists before optimization begins?

In [ ]:
fig, ax = plt.subplots(figsize=(13,4.8))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Region-Constrained Design Theory")
items=[
    ("Physics","what can exist?"),
    ("Admissible Set Ω","what survives?"),
    ("Connected Component Ωc","what remains navigable?"),
    ("Distance Field","what has margin?"),
    ("Operating Point x*","what can operate?"),
    ("Execution","what can be implemented?"),
]
chain_horizontal(ax, items, y=0.48, w=0.135, h=0.20, title_size=9, subtitle_size=7)
ax.text(0.5,0.20,"Optimization selects points. Physics specifies regions.",ha="center",fontsize=15)
savefig(fig,"59_00_region_constrained_design_theory.png")
plt.show()

## 07 Universal Specification Pipeline

Every design problem can be written as resources → constraints → Ω → Ωc → distance → x* → implementation.

In [ ]:
fig, ax = plt.subplots(figsize=(13,4.8))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Universal Specification Pipeline")
items=[
    ("Resources","available states"),
    ("Constraints","failure rules"),
    ("Ω","admissible set"),
    ("Ωc","largest component"),
    ("d(∂Ωc)","distance field"),
    ("x*","arg max margin"),
    ("Implementation","execution"),
]
chain_horizontal(ax, items, y=0.48, w=0.12, h=0.18, title_size=10, subtitle_size=7)
ax.text(0.5,0.19,"Every engineering optimization begins with an admissible region whether or not it is computed explicitly.",ha="center",fontsize=13)
savefig(fig,"59_07_universal_specification_pipeline.png")
plt.show()

## 13 Physics Before Objectives

Objectives summarize a region. They do not create one.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5.8))
for ax in axes:
    clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
axes[0].set_title("Objective-first")
chain_vertical(axes[0],[("Objective","summary"),("Gradient","local direction"),("Minimum","selected point")],x=0.5,y0=0.68,dy=0.17,w=0.52,h=0.09)
axes[0].text(0.5,0.16,"optimizes a point",ha="center",fontsize=12)
axes[1].set_title("Specification-first")
chain_vertical(axes[1],[("Physics","resources"),("Constraints","admissibility"),("Topology","Ωc"),("Robustness","distance"),("Operation","x*")],x=0.5,y0=0.75,dy=0.13,w=0.52,h=0.08)
axes[1].text(0.5,0.16,"specifies the region before the point",ha="center",fontsize=12)
fig.suptitle("Physics Before Objectives",fontsize=19)
fig.text(0.5,0.04,"Objectives summarize a region. They do not create one.",ha="center",fontsize=13)
savefig(fig,"59_13_physics_before_objectives.png")
plt.show()

## 17 Region Geometry

The admissible set has geometry, topology, boundary, holes, and margins.

In [ ]:
fields = {
    "small disconnected": region_field(center=0.55,width=0.13,yscale=0.17,split=True,bump=0.15),
    "connected": region_field(center=0.60,width=0.22,yscale=0.20,bump=0.18),
    "expanded": region_field(center=0.60,width=0.30,yscale=0.23,bump=0.20),
    "robust": region_field(center=0.62,width=0.34,yscale=0.30,bump=0.28),
}
fig, axes = plt.subplots(1,4,figsize=(14,3.9),sharex=True,sharey=True)
for ax,(name,Zi) in zip(axes,fields.items()):
    Mi=metrics(Zi)
    ax.imshow(Zi,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
    ax.contour(X,Y,Zi,levels=[threshold],colors="black",linewidths=1.8)
    ax.contour(X,Y,Mi["largest"].astype(float),levels=[0.5],colors="black",linewidths=2.5)
    ax.scatter([Mi["xstar"]],[Mi["ystar"]],s=80,zorder=4)
    ax.set_title(name)
    ax.set_xlabel("control 1")
    ax.text(0.05,0.88,f"|Ωc|={Mi['largest_area']:.2f}\nM={Mi['max_margin']:.2f}",transform=ax.transAxes,fontsize=9,fontweight="bold")
axes[0].set_ylabel("control 2")
fig.suptitle("Region Geometry: Ω, Ωc, Boundary, Distance",fontsize=18)
savefig(fig,"59_17_region_geometry_progression.png")
plt.show()

## 23 Distance Transform

The distance field inside Ωc measures robustness as distance to failure.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5.5))
Mi = base_M
dist_masked = np.where(Mi["largest"], Mi["dist"], np.nan)
im = ax.imshow(dist_masked, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
ax.contour(X,Y,base_Z,levels=[threshold],colors="black",linewidths=2.5)
ax.scatter([Mi["xstar"]],[Mi["ystar"]],s=160,zorder=4)
ax.annotate("x* = arg max d(∂Ωc)",xy=(Mi["xstar"],Mi["ystar"]),xytext=(0.66,0.20),
            arrowprops=dict(arrowstyle="->",linewidth=2),fontsize=12)
ax.text(0.35,0.25,"Ωc",fontsize=16,fontweight="bold")
ax.set_xlabel("design coordinate 1"); ax.set_ylabel("design coordinate 2")
ax.set_title("Distance Transform: d(∂Ωc) = Distance to Failure")
fig.colorbar(im,ax=ax,label="normalized distance to boundary")
savefig(fig,"59_23_distance_transform_core.png")
plt.show()

## 29 Robust Operation

The same task can have small, medium, or large operating margin depending on the region.

In [ ]:
# Three regions with same nominal x-position but different margins.
fig, axes = plt.subplots(1,3,figsize=(12,3.9),sharex=True,sharey=True)
configs=[("small margin",0.18,0.15),("medium margin",0.24,0.22),("large margin",0.34,0.30)]
for ax,(name,width,yscale) in zip(axes,configs):
    Zi=region_field(center=0.62,width=width,yscale=yscale,bump=0.18)
    Mi=metrics(Zi)
    ax.imshow(Zi,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
    ax.contour(X,Y,Zi,levels=[threshold],colors="black",linewidths=2.0)
    ax.scatter([Mi["xstar"]],[Mi["ystar"]],s=90,zorder=4)
    ax.set_title(name)
    ax.set_xlabel("drive")
    ax.text(0.05,0.86,f"M={Mi['max_margin']:.2f}",transform=ax.transAxes,fontsize=10,fontweight="bold")
axes[0].set_ylabel("loss")
fig.suptitle("Robust Operation: Same Task, Different Margins",fontsize=18)
savefig(fig,"59_29_robust_operation_margins.png")
plt.show()

## 37 Region Comparison

Architecture ranking should compare region topology and robustness before local operating score.

In [ ]:
arch = pd.DataFrame([
    ["Architecture A",0.42,0.36,0.18,0.20,3,2],
    ["Architecture B",0.55,0.51,0.27,0.25,5,4],
    ["Architecture C",0.74,0.70,0.39,0.34,8,7],
    ["Architecture D",0.88,0.84,0.45,0.40,10,8],
],columns=["design","area","largest_component","maximum_margin","average_margin","cost","complexity"])
alpha,beta,gamma,delta=0.55,0.65,0.035,0.025
arch["score"]=alpha*arch["largest_component"]+beta*arch["maximum_margin"]-gamma*arch["cost"]-delta*arch["complexity"]
arch=arch.sort_values("score",ascending=False)
arch.to_csv(DATA/"59_general_architecture_ranking.csv",index=False)

fig, ax = plt.subplots(figsize=(10,4.8))
ypos=np.arange(len(arch))
ax.barh(ypos, arch["score"])
ax.set_yticks(ypos, arch["design"])
ax.invert_yaxis()
ax.set_xlabel("region-constrained score")
ax.set_title("General Architecture Ranking")
for i,v in enumerate(arch["score"]):
    ax.text(v+0.01,i,f"{v:.2f}",va="center")
ax.grid(axis="x",alpha=0.25)
savefig(fig,"59_37_general_architecture_ranking.png")
plt.show()
arch

## 41 Constraint Hierarchy

Physics and constraints specify Ω; topology and distance specify operation.

In [ ]:
fig, ax = plt.subplots(figsize=(11,7))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Constraint Hierarchy")
box(ax,0.34,0.82,0.32,0.08,"Physics","resources",lw=2.4)
constraint_labels=["fabrication","energy","timing","control","loss"]
xs=np.linspace(0.09,0.75,len(constraint_labels))
for lab,x0 in zip(constraint_labels,xs):
    box(ax,x0,0.65,0.16,0.07,lab,"constraint",lw=1.7,title_size=9)
    arrow(ax,(0.50,0.82),(x0+0.08,0.72),lw=1.5)
down=[("Ω","admissible set"),("Ωc","largest connected component"),("d(∂Ωc)","distance to boundary"),("x*","operation")]
ys=[0.47,0.34,0.21,0.08]
for (t,s),y0 in zip(down,ys):
    box(ax,0.33,y0,0.34,0.075,t,s,lw=2.0,title_size=11,subtitle_size=8)
arrow(ax,(0.50,0.65),(0.50,0.545),lw=1.7)
for y1,y2 in zip(ys[:-1],ys[1:]):
    arrow(ax,(0.50,y1),(0.50,y2+0.075),lw=1.7)
savefig(fig,"59_41_constraint_hierarchy.png")
plt.show()

## 47 Search Algorithm

A minimal algorithm for region-constrained design.

In [ ]:
fig, ax = plt.subplots(figsize=(7,7.5))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Region-Constrained Search Algorithm")
steps=[
    ("simulate","physics model"),
    ("construct Ω","threshold constraints"),
    ("extract Ωc","connected components"),
    ("compute d(∂Ωc)","distance transform"),
    ("select x*","arg max margin"),
    ("return design","implementation"),
]
chain_vertical(ax,steps,x=0.5,y0=0.78,dy=0.12,w=0.50,h=0.075,title_size=11)
savefig(fig,"59_47_search_algorithm.png")
plt.show()

## 53 Applications

The same pipeline applies across domains.

In [ ]:
apps=pd.DataFrame([
    ["Quantum hardware","gate resources","coherence, loss","fault-tolerant operation"],
    ["Robotics","reachable states","collision, torque","safe trajectory"],
    ["Controls","stable states","gain, delay","robust controller"],
    ["Materials","candidate phases","stress, defects","manufacturable material"],
    ["Machine learning","model family","calibration, shift","robust predictor"],
    ["Networking","routes","capacity, latency","reliable communication"],
],columns=["Domain","Resources","Constraints","Operation"])
apps.to_csv(DATA/"59_applications_table.csv",index=False)

fig, ax = plt.subplots(figsize=(12,4.4))
clean_axes(ax)
ax.set_title("Applications Share the Same Region Pipeline")
tbl=ax.table(cellText=apps.values,colLabels=apps.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1,1.55)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"59_53_applications_table.png")
plt.show()
apps

## 59 Research Program

The framework becomes a research program linking mathematics, engineering, and science.

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,4.8))
cols=[
    ("Mathematics",[("Topology","connected components"),("Distance Transform","boundary margin"),("Geometry","region shape")]),
    ("Engineering",[("Robust Design","safe margin"),("Architecture Search","rank by Ωc"),("Calibration","preserve region")]),
    ("Science",[("Physics","resources"),("Simulation","construct Ω"),("Experiment","validate Ωc")]),
]
for ax,(title,items) in zip(axes,cols):
    clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title(title)
    for i,(t,s) in enumerate(items):
        box(ax,0.12,0.68-i*0.22,0.76,0.12,t,s,lw=1.9,title_size=11)
fig.suptitle("Region-Constrained Design Theory as a Research Program",fontsize=18)
fig.text(0.5,0.06,"Physics specifies. Mathematics describes. Engineering implements.",ha="center",fontsize=13)
savefig(fig,"59_59_research_program.png")
plt.show()

## 60 General Design Principle

A compact closing figure.

In [ ]:
fig, ax = plt.subplots(figsize=(11,5.8))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("General Design Principle")
ax.text(0.5,0.72,"Compute the region",ha="center",fontsize=24,fontweight="bold")
ax.text(0.5,0.58,"preserve its connected topology",ha="center",fontsize=18)
ax.text(0.5,0.47,"then",ha="center",fontsize=14)
ax.text(0.5,0.36,"select the operating point",ha="center",fontsize=18)
items=[("Ω","admissible"),("Ωc","connected"),("d(∂Ωc)","robust"),("x*","operation")]
chain_horizontal(ax,items,y=0.12,x0=0.18,x1=0.82,w=0.14,h=0.10,title_size=12,subtitle_size=8)
savefig(fig,"59_60_general_design_principle_manifesto.png")
plt.show()

## 67 Export summary metadata

In [ ]:
summary = {
    "notebook": "59_general_region_constrained_design_theory",
    "thesis": "Optimization selects points. Physics specifies regions.",
    "pipeline": ["Resources", "Constraints", "Ω", "Ωc", "d(∂Ωc)", "x*", "Implementation"],
    "principles": [
        "Objectives summarize a region; they do not create one.",
        "The admissible set Ω is specified before optimization begins.",
        "The largest connected component Ωc is the navigable design family.",
        "The distance transform d(∂Ωc) measures robustness.",
        "The operating point x* is selected after topology is preserved."
    ],
    "figure_count": len(list(FIG.glob("*.png"))),
    "data_files": sorted([p.name for p in DATA.glob("*")]),
}
(DATA/"59_design_theory_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

## 71 Package and download outputs

In [ ]:
# ============================================================
# Package and download Notebook 59 outputs
# ============================================================
#
# Run this final cell after executing the notebook.
#
# It creates:
#   59_general_region_constrained_design_theory_outputs.zip
#
# containing:
#   figures/*.png
#   data/*.csv
#   data/*.json
#   README_59_outputs.md
#
# In Google Colab, it triggers an automatic browser download.
# In Jupyter, it displays a clickable download link.

package_root = Path("outputs/notebook_59_region_constrained_design_theory")
figure_dir = package_root / "figures"
data_dir = package_root / "data"

figure_files = sorted(figure_dir.glob("*.png"))
data_files = sorted([p for p in data_dir.glob("*") if p.is_file()])

manifest = package_root / "README_59_outputs.md"
manifest.write_text(
    "# Notebook 59 Outputs\n\n"
    "Generated from `59_general_region_constrained_design_theory.ipynb`.\n\n"
    "## Figures\n"
    + ("\n".join(f"- figures/{p.name}" for p in figure_files) if figure_files else "- No figures found. Run all cells before packaging.")
    + "\n\n## Data\n"
    + ("\n".join(f"- data/{p.name}" for p in data_files) if data_files else "- No data files found. Run all cells before packaging.")
    + "\n",
    encoding="utf-8",
)

archive_path = Path(shutil.make_archive("59_general_region_constrained_design_theory_outputs", "zip", root_dir=package_root)).resolve()

print(f"Packaged outputs: {archive_path}")
print(f"Figures: {len(figure_files)}")
print(f"Data files: {len(data_files)}")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    display(FileLink(str(archive_path)))